In [5]:
import numpy as np
import pandas as pd
from collections import defaultdict
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score,precision_recall_fscore_support,classification_report,confusion_matrix
)
df=pd.read_csv("pos_tags.csv",encoding="latin-1")
print("Dataset shape:",df.shape)
print(df.head())
df.columns=['Sentence','Word','Tag']
df['Sentence']=df['Sentence'].ffill()
print("\nMissing Values")
print(df.isnull().sum())
sentences=[]
grouped=df.groupby("Sentence")
for sentence_id,group in grouped:
    words=list(group["Word"])
    pos_tags=list(group["Tag"])
    sentences.append((words,pos_tags))
print("\nTotal Sentences:",len(sentences))
print("\nExample Sentence")
print(sentences[0])
train_data,test_data=train_test_split(sentences,test_size=0.2,random_state=42,shuffle=True)
print("\nTraining Sentences:",len(train_data))
print("\nTesting sentences:",len(test_data))
vocab=set()
tags=set()
for words,pos in train_data:
    vocab.update(words)
    tags.update(pos)
vocab=sorted(list(vocab))
tags=sorted(list(tags))
print("\nVocabulary size:",len(vocab))
print("Number of POS Tags:",len(tags))
word2idx={}
idx2word={}
for i,word in enumerate(vocab):
    word2idx[word]=i
    idx2word[i]=word
tag2idx={}
idx2tag={}
for i,tag in enumerate(tags):
    tag2idx[tag]=i
    idx2tag[i]=tag
print("\nSample Word Mapping")
for word in list(word2idx.keys())[:10]:
    print(word,"->",word2idx[word])
print("\nTag Mapping")
for tag in tags:
    print(tag,"=>",tag2idx[tag])
print("\n=====================")
print("Data Summary")
print("=========================")
print("Training Sentences:",len(train_data))
print("Testing Sentences:",len(test_data))
print("Vocabulary Size:",len(vocab))
print("Number of Tags:",len(tags))
print("\nAvailable Tags")
print(tags)
print("="*60)
print("Building hmm probabitlity matrices")
print("="*60)
num_tags=len(tags)
num_words=len(vocab)
print("Number of Tags:",num_tags)
print("Vocabulary Size:",num_words)
initial_counts=np.ones(num_tags)
for words,pos_tags in train_data:
    first_tag=pos_tags[0]
    initial_counts[tag2idx[first_tag]]+=1
initial_probs=initial_counts/initial_counts.sum()
print("\nInitial Probability Matrix Shape")
print(initial_probs.shape)
transition_counts=np.ones((num_tags,num_tags))
for words,pos_tags in train_data:
    for i in range(len(pos_tags)-1):
        current_tag=tag2idx[pos_tags[i]]
        next_tag=tag2idx[pos_tags[i+1]]
        transition_counts[current_tag,next_tag]+=1
transition_probs=transition_counts/transition_counts.sum(axis=1,keepdims=True)
print("\nTransition Matrix Shape")
print(transition_probs.shape)
emission_counts=np.ones((num_tags,num_words))
for words,pos_tags in train_data:
    for word,tag in zip(words,pos_tags):
        word_index=word2idx[word]
        tag_index=tag2idx[tag]
        emission_counts[tag_index,word_index]+=1
emission_probs=emission_counts/emission_counts.sum(axis=1,keepdims=True)
print("\nEmission Matrix shape")
print(emission_probs.shape)
print("\nChecking Probability sums")
print("Initial Sum:",initial_probs.sum())
print("Transition Row 0:",transition_probs[0].sum())
print("Emission Row 0:",emission_probs[0].sum())
log_initial=np.log(initial_probs)
log_transition=np.log(transition_probs)
log_emission=np.log(emission_probs)
print("\nConverted all matrices to log space")
unknown_probability=1e-10
log_unknown=np.log(unknown_probability)
print("Unknown Word Log Probability:",log_unknown)
print("\nSample initial Probabilites")
for tag in tags[:10]:
    index=tag2idx[tag]
    print(f"{tag:5s}:{initial_probs[index]:.6f}")
print("\nSample transition probabilites")
for i in range(min(5,num_tags)):
    current=idx2tag[i]
    print("\nCurrent tag:",current)
    for j in range(min(5,num_tags)):
        nxt=idx2tag[j]
        print(f"{current}->{nxt}={transition_probs[i,j]:.6f}")
print("\nSample Emission Probabilites")
for tag in tags[:5]:
    tag_index=tag2idx[tag]
    print("\nTag:",tag)
    non_zero=np.argsort(emission_probs[tag_index])[::-1][:5]
    for word_index in non_zero:
        print(idx2word[word_index],"->",emission_probs[tag_index,word_index])
print("\nMemory Usage:")
print("Emission Matrix:",round(emission_probs.nbytes/(1024*1024),2),"MB")
print("\nTraining Completed sucessfully")
print("="*60)
print("Implementing viterbi algorithm")
print("="*60)
def viterbi(sentence):
    """Predict POS tags for a sentence using the HMM"""
    T=len(sentence)
    N=len(tags)
    dp=np.full((N,T),-np.inf)
    backpointer=np.zeros((N,T),dtype=int)
    first_word=sentence[0]
    if first_word in word2idx:
        emission=log_emission[:,word2idx[first_word]]
    else:
        emission=np.full(N,log_unknown)
    dp[:,0]=log_initial+emission
    for t in range(1,T):
        word=sentence[t]
        if word in word2idx:
            emission=log_emission[:,word2idx[word]]
        else:
            emission=np.full(N,log_unknown)
        scores=dp[:,t-1][:,np.newaxis]+log_transition
        backpointer[:,t]=np.argmax(scores,axis=0)
        dp[:,t]=np.max(scores,axis=0)+emission
    best_path=np.zeros(T,dtype=int)
    best_path[-1]=np.argmax(dp[:,T-1])
    for t in range(T-2,-1,-1):
        best_path[t]=backpointer[best_path[t+1],t+1]
    predicted_tags=[idx2tag[i] for i in best_path]
    return predicted_tags
sample_words,sample_tags=train_data[0]
prediction=viterbi(sample_words)
print("\nSample Sentence")
print(sample_words)
print("\nActual Tags")
print(sample_tags)
print("\nPredicted Tags")
print(prediction)
true_tags=[]
predicted_tags=[]
print("\nPredicting Test dataset....")
for words,actual in test_data:
    predicted=viterbi(words)
    true_tags.extend(actual)
    predicted_tags.extend(predicted)
print("Prediction Completed")
print("\nSample Predictions")
for i in range(5):
    words,actual=test_data[i]
    pred=viterbi(words)
    print("\nSentence",i+1)
    print("-"*50)
    for w,a,p in zip(words,actual,pred):
        print(f"{w:15s} Actual:{a:6s} Predicted:{p}")
print("\nPrediction Summary")
print("Total Actual tags:",len(true_tags))
print("Total Predicted Tags:",len(predicted_tags))
correct=sum(1 for a,p in zip(true_tags,predicted_tags) if a==p)
print("Correct Predictions:",correct)
print("Incorrect Predictions:",len(true_tags)-correct)

print("Predictin Accuracy(raw):",round(correct/len(true_tags),4))
print("\nVectorized Viterbi Algorith Completed Successfully")
print("="*60)
print("Model Evaluation")
print("="*60)
accuracy=accuracy_score(true_tags,predicted_tags)
print(f"\nAccuracy:{accuracy:.4f}")
precision,recall,f1,_=precision_recall_fscore_support(true_tags,predicted_tags,average='weighted',zero_division=0)
print(f"Precision:{precision:.4f}")
print(f"Recall:{recall:.4f}")
print(f"F1 Score:{f1:.4f}")
print("\n")
print("="*60)
print("Classification Report")
print("="*60)
print(classification_report(true_tags,predicted_tags,zero_division=0))
print("=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)
labels=sorted(list(set(true_tags)))
cm=confusion_matrix(true_tags,predicted_tags,labels=labels)
print("Matrix Shape:",cm.shape)
print(cm)
print("\n")
print("="*60)
print("Testing on unseen sentences")
print("="*60)
test_sentences = [

    ["I", "love", "machine", "learning"],

    ["She", "is", "reading", "a", "book"],

    ["The", "dog", "runs", "fast"],

    ["OpenAI", "creates", "powerful", "models"],

    ["Students", "study", "Python", "daily"]

]
for i,sentence in enumerate(test_sentences):
    prediction=viterbi(sentence)
    print(f"\nSentence{i+1}")
    print("-"*50)
    for word,tag in zip(sentence,prediction):
        print(f"{word:15s}-->{tag}")
print("\n")
print("="*60)
print("Analysis")
print("="*60)
analysis = [
    "1. The HMM correctly predicts common grammatical patterns such as Determiner → Noun → Verb.",
    "2. Unknown words (e.g., 'OpenAI', 'Python') are tagged using transition probabilities because they are absent from the training vocabulary.",
    "3. Verbs following pronouns are usually predicted correctly due to strong transition probabilities.",
    "4. Adjectives preceding nouns are generally identified accurately.",
    "5. Some errors may occur for ambiguous words because a first-order HMM only considers the previous tag."
]
for line in analysis:
    print(line)
print("\n")
print("="*60)
print("Final Summary")
print("="*60)
print(f"Training Sentences:{len(train_data)}")
print(f"Testing Sentences:{len(test_data)}")
print(f"Vocabulary Size:{len(vocab)}")
print(f"POS Tags:{len(tags)}")
print(f"Accuracy:{accuracy:.4f}")
print(f"Precision:{precision:.4f}")
print(f"Recall:{recall:.4f}")
print(f"F1 Score:{f1:.4f}")

print("\nHMM POS Tagger Completed Successfully!")

Dataset shape: (370100, 3)
   sentence_id    word  tag
0            0      aa   NN
1            1     aaa   NN
2            2     aah   NN
3            3   aahed  VBN
4            4  aahing  VBG

Missing Values
Sentence    0
Word        0
Tag         0
dtype: int64

Total Sentences: 370100

Example Sentence
(['aa'], ['NN'])

Training Sentences: 296080

Testing sentences: 74020

Vocabulary size: 296080
Number of POS Tags: 24

Sample Word Mapping
FALSE -> 0
TRUE -> 1
aa -> 2
aaa -> 3
aahed -> 4
aahing -> 5
aahs -> 6
aal -> 7
aaliis -> 8
aals -> 9

Tag Mapping
CC => 0
CD => 1
DT => 2
IN => 3
JJ => 4
JJR => 5
JJS => 6
MD => 7
NN => 8
NNS => 9
PRP => 10
PRP$ => 11
RB => 12
RBR => 13
VB => 14
VBD => 15
VBG => 16
VBN => 17
VBP => 18
VBZ => 19
WDT => 20
WP => 21
WP$ => 22
WRB => 23

Data Summary
Training Sentences: 296080
Testing Sentences: 74020
Vocabulary Size: 296080
Number of Tags: 24

Available Tags
['CC', 'CD', 'DT', 'IN', 'JJ', 'JJR', 'JJS', 'MD', 'NN', 'NNS', 'PRP', 'PRP$', 'RB', 'RBR'